# ☀️ Solar Panel Recommendation ML

Clean portfolio notebook based on the original DS323 project.

The goal is to classify a household's **solar-investment likelihood** as:

- `0` — Low
- `1` — Moderate
- `2` — High

The project uses synthetic household energy, income, and emissions data.

> **Important:** `buy_decision` is an engineered proxy target created from household
> financial, energy-consumption, and environmental variables. It is not an observed
> real-world customer purchase outcome.


## 1. Imports


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, QuantileTransformer
from sklearn.svm import LinearSVC, SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

SEED = 42
np.random.seed(SEED)


## 2. Load the Dataset

Update `DATA_PATH` if the dataset is stored somewhere else.


In [ ]:
DATA_PATH = "data/synthetic_energy_dataset.csv"

df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
display(df.head())


## 3. Feature Engineering — `buy_decision`

The original project created the target from a scoring rule based on:

- energy cost relative to monthly income,
- household energy consumption,
- CO₂ emissions.


In [ ]:
def calculate_buy_decision(row):
    score = 0

    # Financial factor
    savings_potential = row["EnergyCost_USD"] / row["MonthlyIncome_USD"]
    if savings_potential > 0.05:
        score += 3
    elif savings_potential > 0.02:
        score += 1

    # Energy-consumption factor
    if row["EnergyConsumed_kWh"] > 500:
        score += 2
    elif row["EnergyConsumed_kWh"] > 300:
        score += 1

    # Environmental factor
    if row["CO2Emissions_kg"] > 100:
        score += 1

    if score >= 4:
        return 2
    elif score >= 2:
        return 1
    return 0


df["buy_decision"] = df.apply(calculate_buy_decision, axis=1)

print(df["buy_decision"].value_counts().sort_index())


## 4. Basic Data Quality Checks


In [ ]:
print("Missing values:", int(df.isna().sum().sum()))
print("Duplicate rows:", int(df.duplicated().sum()))

numeric_features = [
    "EnergyConsumed_kWh",
    "SolarEnergyGenerated_kWh",
    "EnergyCost_USD",
    "MonthlyIncome_USD",
    "ApplianceUsage_Hours",
    "CO2Emissions_kg",
]

def iqr_outlier_count(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return int(((series < lower) | (series > upper)).sum())

outlier_summary = {
    col: iqr_outlier_count(df[col])
    for col in numeric_features
}

pd.Series(outlier_summary, name="IQR_outlier_count")


## 5. Modeling Dataset

To match the deployed prototype, the model uses three user-facing inputs:

- Monthly Income
- Energy Consumed
- CO₂ Emissions

The original project sampled 8,000 rows for model experiments.


In [ ]:
df_sample = df.sample(n=min(8000, len(df)), random_state=SEED)

selected_features = [
    "MonthlyIncome_USD",
    "EnergyConsumed_kWh",
    "CO2Emissions_kg",
]

X = df_sample[selected_features]
y = df_sample["buy_decision"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=SEED,
)

print("Train:", X_train.shape)
print("Test :", X_test.shape)


## 6. Scaling

The original experiments compared **StandardScaler** and
**QuantileTransformer**.


In [ ]:
standard_scaler = StandardScaler()
X_train_standard = standard_scaler.fit_transform(X_train)
X_test_standard = standard_scaler.transform(X_test)

quantile_scaler = QuantileTransformer(
    output_distribution="normal",
    random_state=SEED,
)
X_train_quantile = quantile_scaler.fit_transform(X_train)
X_test_quantile = quantile_scaler.transform(X_test)


## 7. Baseline Models


In [ ]:
baseline_models = {
    "SVM": (
        LinearSVC(dual=False, random_state=SEED, max_iter=1000),
        X_train,
        X_test,
    ),
    "Naive Bayes": (
        GaussianNB(),
        X_train,
        X_test,
    ),
    "MLP": (
        MLPClassifier(random_state=SEED, max_iter=300),
        X_train,
        X_test,
    ),
    "Random Forest": (
        RandomForestClassifier(random_state=SEED),
        X_train,
        X_test,
    ),
}

baseline_rows = []

for name, (model, Xtr, Xte) in baseline_models.items():
    model.fit(Xtr, y_train)
    train_pred = model.predict(Xtr)
    test_pred = model.predict(Xte)

    baseline_rows.append({
        "Model": name,
        "Training Accuracy": accuracy_score(y_train, train_pred),
        "Testing Accuracy": accuracy_score(y_test, test_pred),
    })

baseline_results = pd.DataFrame(baseline_rows)
baseline_results


## 8. Hyperparameter Tuning

The same parameter grids used in the original project are retained here.


In [ ]:
param_grid_svm = {
    "C": [0.001, 0.01, 0.1, 1, 10],
    "gamma": [0.001, 0.01, 0.1, 1, 10],
}

param_grid_mlp = {
    "hidden_layer_sizes": [(50,), (100,)],
    "activation": ["relu"],
    "solver": ["adam"],
    "alpha": [0.0001, 0.001],
    "learning_rate": ["constant"],
}

param_grid_rf = {
    "n_estimators": [50, 100],
    "max_depth": [10, 15],
}

param_grid_nb = {
    "var_smoothing": [1e-9, 1e-8, 1e-7, 1e-6, 1e-5],
}


In [ ]:
svm_grid = GridSearchCV(
    SVC(probability=True),
    param_grid_svm,
    cv=3,
    n_jobs=-1,
)
svm_grid.fit(X_train_standard, y_train)

mlp_grid = GridSearchCV(
    MLPClassifier(
        random_state=SEED,
        max_iter=500,
        early_stopping=True,
    ),
    param_grid_mlp,
    cv=3,
    n_jobs=-1,
)
mlp_grid.fit(X_train_quantile, y_train)

rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=SEED),
    param_grid_rf,
    cv=3,
    n_jobs=-1,
)
rf_grid.fit(X_train, y_train)

nb_grid = GridSearchCV(
    GaussianNB(),
    param_grid_nb,
    cv=3,
    n_jobs=-1,
)
nb_grid.fit(X_train_quantile, y_train)

print("Best SVM parameters:", svm_grid.best_params_)
print("Best MLP parameters:", mlp_grid.best_params_)
print("Best Random Forest parameters:", rf_grid.best_params_)
print("Best Naive Bayes parameters:", nb_grid.best_params_)


## 9. Tuned Model Evaluation

**Portfolio cleanup note:** the original notebook trained the tuned MLP on
QuantileTransformer data but evaluated it on StandardScaler data in one cell.
This cleaned version evaluates it with the same QuantileTransformer used
during training.


In [ ]:
tuned_models = {
    "SVM": (
        svm_grid.best_estimator_,
        X_train_standard,
        X_test_standard,
    ),
    "MLP": (
        mlp_grid.best_estimator_,
        X_train_quantile,
        X_test_quantile,
    ),
    "Random Forest": (
        rf_grid.best_estimator_,
        X_train,
        X_test,
    ),
    "Naive Bayes": (
        nb_grid.best_estimator_,
        X_train_quantile,
        X_test_quantile,
    ),
}

evaluation_rows = []
predictions = {}

for name, (model, Xtr, Xte) in tuned_models.items():
    model.fit(Xtr, y_train)

    train_pred = model.predict(Xtr)
    test_pred = model.predict(Xte)
    predictions[name] = test_pred

    evaluation_rows.append({
        "Model": name,
        "Train Accuracy": accuracy_score(y_train, train_pred),
        "Test Accuracy": accuracy_score(y_test, test_pred),
        "Precision (weighted)": precision_score(
            y_test, test_pred, average="weighted", zero_division=0
        ),
        "Recall (weighted)": recall_score(
            y_test, test_pred, average="weighted", zero_division=0
        ),
        "F1 (weighted)": f1_score(
            y_test, test_pred, average="weighted", zero_division=0
        ),
    })

tuned_results = pd.DataFrame(evaluation_rows).sort_values(
    "Test Accuracy",
    ascending=False,
)

tuned_results


## 10. Classification Reports


In [ ]:
for name, pred in predictions.items():
    print(f"\n{name}")
    print("-" * len(name))
    print(classification_report(
        y_test,
        pred,
        target_names=["Low", "Moderate", "High"],
        zero_division=0,
    ))


## 11. Confusion Matrix — Tuned MLP

The project deployed an MLP-based Streamlit prototype, so the confusion matrix
below focuses on the tuned MLP.


In [ ]:
mlp_pred = predictions["MLP"]
cm = confusion_matrix(y_test, mlp_pred)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm)

ax.set_title("Confusion Matrix — Tuned MLP")
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")

labels = ["Low", "Moderate", "High"]
ax.set_xticks(range(3), labels)
ax.set_yticks(range(3), labels)

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, cm[i, j], ha="center", va="center")

fig.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()


## 12. Optional Export for Deployment

This saves the tuned MLP together with the **matching QuantileTransformer**
and feature list so preprocessing stays consistent at inference time.


In [ ]:
import pickle

deployment_bundle = {
    "model": mlp_grid.best_estimator_,
    "scaler": quantile_scaler,
    "features": selected_features,
    "class_labels": {
        0: "Low",
        1: "Moderate",
        2: "High",
    },
}

with open("solar_recommender_bundle.pkl", "wb") as f:
    pickle.dump(deployment_bundle, f)

print("Saved: solar_recommender_bundle.pkl")


## Portfolio Scope

This public notebook focuses on the implemented machine-learning workflow:
target engineering, data quality checks, preprocessing, model comparison,
hyperparameter tuning, evaluation, and deployment preparation.

The associated Streamlit application can be linked from the GitHub README as
a live demo.
